# LogSeer Prediction
---

Predicts whether a planned operation will succeed or fail based on current server logs.

**Setup**: copy your data to `DATA_DIR`. Two layouts are supported:

```
# Single prediction — log files directly in DATA_DIR:
DATA_DIR/
├── log1.txt
└── log2.txt

# Multiple predictions — one subdirectory per operation run:
DATA_DIR/
├── 0001/
│   ├── log1.txt
│   └── log2.txt
└── 0002/
    └── log1.txt
```

You only need the model files for the models you want to run — the NN and sklearn model are each optional. For the NN you need `logseer.keras` and `tokenizer.pickle`; for an sklearn model just the `.pkl` file. If you already ran training on this machine the files are already in the repo root. Otherwise copy them manually or use the Colab cell below to load them from Google Drive.

**Output**: each row in the results is one operation run, with `OR` and `AND` ensemble predictions — use `AND` as the high-precision signal.

> **⚡ Start here** — run all cells from top to bottom. Skip any marked **COLAB ONLY** if running on a self-hosted environment.

In [ ]:
# Setup
import os, sys

# On Google Colab: clone the repo and set up the path
# On local: assumes notebook is run from the notebooks/ directory

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if not os.path.exists('logseer'):
    os.system('git clone https://github.com/masahiko-shibata/logseer.git')
    os.chdir('logseer')

sys.path.insert(0, '.')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from logseer import Seer, print_results

In [ ]:
# Configuration
MODEL_BASE_DIR            = './'                           # directory where trained models are stored
DATA_DIR            = 'data_current'
NN_MODEL_FILE       = 'logseer.keras'
TOKENIZER_FILE      = 'tokenizer.pickle'
SK_MODEL_FILE       = 'xgb.pkl'
NN_MODEL_PATH       = MODEL_BASE_DIR + NN_MODEL_FILE
TOKENIZER_PATH      = MODEL_BASE_DIR + TOKENIZER_FILE
SK_MODEL_PATH       = MODEL_BASE_DIR + SK_MODEL_FILE
NUMCHAR             = 3000
MAX_SEQUENCE_LENGTH = 26000
NN_THRESHOLD        = 0.72
XGB_THRESHOLD       = 0.52

---
### ☁️ COLAB ONLY — run the cell below to copy data and models from Google Drive

> **Model files** — upload the model files you trained to `My Drive/Colab Notebooks/logseer/models/`. You only need the files for the models you want to run: `logseer.keras` + `tokenizer.pickle` for the NN, or your sklearn model file (e.g. `xgb.pkl`, `rf.pkl`) on its own. Missing files are skipped automatically.
>
> **Data** — zip your `data_current/` folder as `data_current.zip` (the zip should extract to a `data_current/` folder, not its contents directly) and upload it to `My Drive/Colab Notebooks/logseer/data/`.
>
> **Skip if running on a self-hosted environment** — copy model files to the repo root and data to `DATA_DIR` manually.

---

In [ ]:
# Copy data and trained models from Google Drive.
from google.colab import drive
import shutil, zipfile, os
drive.mount('/content/drive')

DRIVE_BASE      = '/content/drive/MyDrive/Colab Notebooks/logseer'
DRIVE_DATA_DIR  = DRIVE_BASE + '/data/'
DRIVE_MODEL_DIR = DRIVE_BASE + '/models/'
DATA_ZIP        = 'data_current.zip'

for src, dst in [
    (DRIVE_MODEL_DIR + NN_MODEL_FILE,  NN_MODEL_PATH),
    (DRIVE_MODEL_DIR + TOKENIZER_FILE, TOKENIZER_PATH),
    (DRIVE_MODEL_DIR + SK_MODEL_FILE,  SK_MODEL_PATH),
    (DRIVE_DATA_DIR  + DATA_ZIP,       DATA_ZIP),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied: {os.path.basename(src)}')
    else:
        print(f'Not found, skipping: {os.path.basename(src)}')

if os.path.exists(DATA_ZIP):
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        z.extractall('.')
    print(f'Extracted {DATA_ZIP}')

print(os.listdir())

> At this point your data and models are in place. Run the cell below to load models and predict.

In [ ]:
# Load models and predict
seer    = Seer.from_files(NN_MODEL_PATH, TOKENIZER_PATH, SK_MODEL_FILE,
                          numchar=NUMCHAR, max_sequence_length=MAX_SEQUENCE_LENGTH,
                          nn_threshold=NN_THRESHOLD, xgb_threshold=XGB_THRESHOLD)
results = seer.predict(DATA_DIR)
print_results(results)